In [ ]:
!pip install -q groq pandas tqdm

from groq import Groq
from google.colab import userdata, files
import pandas as pd
from tqdm.auto import tqdm
import time


In [ ]:
from google.colab import files

print("Upload your two files:")
uploaded = files.upload()



In [ ]:

ORIGINAL_FILE = '/content/revised_original_dataset.csv'
DETECTION_FILE = '/content/result_detection.csv'

df_orig = pd.read_csv(ORIGINAL_FILE)
df_det = pd.read_csv(DETECTION_FILE)


df = df_orig.merge(df_det[['qid', 'selfcheck_halluc_score', 'is_flagged']], on='qid', how='left')
df['is_flagged'] = df['is_flagged'].fillna(False)

print("Total questions:", len(df))
print("Flagged for mitigation:", df['is_flagged'].sum())

flagged_df = df[df['is_flagged']].copy().reset_index(drop=True)
print("\nQuestions to mitigate:", len(flagged_df))
display(flagged_df[['qid', 'question', 'generated_answer', 'selfcheck_halluc_score']].head(3))

In [ ]:
time.sleep(1.5)

In [ ]:
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
MODEL = "llama-3.3-70b-versatile"

def run_cove(question, baseline_code):
    try:
        # Step 1: Plan verification questions
        plan_prompt = f"""\
Given this coding problem and draft solution, create 5-7 short, independent verification questions to check correctness.
Focus on: syntax validity, logic correctness, edge cases, input/output handling, Python/JS standard behavior, efficiency claims.
Do not assume the draft is correct.

Problem: {question}
Draft code:
{baseline_code}

Output only a numbered list of questions, nothing else.
"""

        plan_resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": plan_prompt}],
            temperature=0.3,
            max_tokens=600
        )
        verif_text = plan_resp.choices[0].message.content.strip()


        verif_answers = []
        questions_list = [line.strip() for line in verif_text.split('\n') if line.strip() and line[0].isdigit()]

        for q in questions_list:

            q_clean = q.split('.', 1)[1].strip() if '.' in q else q
            ans_prompt = f"Answer this question concisely and factually about Python/JavaScript coding:\n{q_clean}"

            ans_resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": ans_prompt}],
                temperature=0.1,
                max_tokens=400
            )
            answer = ans_resp.choices[0].message.content.strip()
            verif_answers.append(f"Q: {q_clean}\nA: {answer}")
            time.sleep(1)

        verif_combined = "\n\n".join(verif_answers)


        revise_prompt = f"""\
Using ONLY the verified answers below (ignore contradictions in the original draft), write a correct, concise version of the code.
Do not add explanations — output code only.

Original problem: {question}
Original draft:
{baseline_code}

Verified facts:
{verif_combined}
"""

        revise_resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": revise_prompt}],
            temperature=0.2,
            max_tokens=1500
        )

        corrected_code = revise_resp.choices[0].message.content.strip()
        return corrected_code

    except Exception as e:
        return f"CoVe ERROR: {str(e)}"



In [ ]:

flagged_df['cove_final_answer'] = None
flagged_df['cove_status'] = 'pending'

for idx, row in tqdm(flagged_df.iterrows(), total=len(flagged_df), desc="Applying CoVe"):
    print(f"Processing {row['qid']}...")
    try:
        corrected = run_cove(row['question'], row['generated_answer'])
        flagged_df.at[idx, 'cove_final_answer'] = corrected
        flagged_df.at[idx, 'cove_status'] = 'success'
    except Exception as e:
        flagged_df.at[idx, 'cove_final_answer'] = str(e)
        flagged_df.at[idx, 'cove_status'] = 'failed'
        print(f"  → Failed: {str(e)}")

    if (idx + 1) % 5 == 0:
        flagged_df.to_csv('/content/results_mitigation_partial.csv', index=False)
        print("  Partial save done")


flagged_df.to_csv('/content/results_mitigation.csv', index=False)
print("\nCoVe mitigation complete.")
display(flagged_df[['qid', 'question', 'generated_answer', 'cove_final_answer', 'cove_status']].head())

files.download('/content/results_mitigation.csv')

In [ ]:
mitigated = pd.read_csv('/content/results_mitigation.csv')

success = (mitigated['cove_status'] == 'success').sum()
failed = (mitigated['cove_status'].str.contains('ERROR', na=False)).sum()

print(f"Success: {success} / {len(mitigated)}")
print(f"Failed: {failed}")
print("\nFailed rows:")
display(mitigated[mitigated['cove_status'].str.contains('ERROR', na=False)][['qid', 'question', 'cove_final_answer']])

In [ ]:
import pandas as pd
from google.colab import files

mitigated = pd.read_csv('/content/results_mitigation.csv')

def clean_code(text):
    if pd.isna(text):
        return text
    text = str(text).strip()
    if text.startswith('```'):
        lines = text.split('\n')

        if lines[0].startswith('```') and (len(lines) < 2 or lines[-1].startswith('```')):
            text = '\n'.join(lines[1:-1]).strip()
        elif lines[0].startswith('```'):
            text = '\n'.join(lines[1:]).strip()
    return text

mitigated['cove_clean_code'] = mitigated['cove_final_answer'].apply(clean_code)


mitigated.to_csv('/content/results_mitigation_clean.csv', index=False)
print("Cleaned file saved. Columns include: cove_clean_code (without ``` fences)")
files.download('/content/results_mitigation_clean.csv')

In [ ]:

sample = mitigated.sample(6) if len(mitigated) > 6 else mitigated.head(6)

for _, row in sample.iterrows():
    print(f"\n=== {row['qid']} ===")
    print("Question:", row['question'][:100] + "..." if len(row['question']) > 100 else row['question'])
    print("\nBaseline:\n", row['generated_answer'][:300] + "..." if len(row['generated_answer']) > 300 else row['generated_answer'])
    print("\nCoVe version:\n", row['cove_final_answer'][:500] + "..." if len(row['cove_final_answer']) > 500 else row['cove_final_answer'])
    print("-" * 80)

VISUALIZATION

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files


orig = pd.read_csv('revised_original_dataset.csv')
detect = pd.read_csv('result_detection.csv')
mit = pd.read_csv('results_mitigation.csv')

df = orig.merge(detect[['qid', 'selfcheck_halluc_score', 'is_flagged']], on='qid')
df = df.merge(mit[['qid', 'cove_final_answer', 'cove_status']], on='qid', how='left')

print("Dataset ready:", df.shape)


plt.figure(figsize=(12, 6))
sns.histplot(df['selfcheck_halluc_score'].dropna(), bins=20, kde=True, color='#e74c3c')
plt.axvline(0.5, color='darkred', ls='--', label='Threshold (0.5)')
plt.title('SelfCheckGPT Hallucination Scores – 30 Coding Questions\n'
          f'Flagged: {df["is_flagged"].sum()}/30 ({df["is_flagged"].mean()*100:.0f}%)', fontsize=14)
plt.xlabel('Hallucination Score (higher = more inconsistent)')
plt.ylabel('Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('selfcheck_scores.png', dpi=200)
plt.show()
files.download('selfcheck_scores.png')


status = df['cove_status'].value_counts(dropna=True)
plt.figure(figsize=(7, 5))
status.plot(kind='bar', color=['#2ecc71', '#e74c3c', '#95a5a6'])
plt.title('CoVe Processing Outcome on Flagged Questions', fontsize=14)
plt.ylabel('Number of questions')
plt.xticks(rotation=0)
plt.savefig('cove_status.png', dpi=200)
plt.show()
files.download('cove_status.png')


print("\nBefore vs After (random 5 successful):")
sample = df[df['cove_status'] == 'success'].sample(5)
display(sample[['qid', 'question', 'generated_answer', 'cove_final_answer']])

UPLOADING TO GITHUB

In [ ]:
!ls -l /content/

In [ ]:
!pip install nbconvert -q

NOTEBOOK_NAME = "SelfCheckGPT.ipynb"

!jupyter nbconvert --ClearOutputPreprocessor.enabled=True \
                   --ClearMetadataPreprocessor.enabled=True \
                   --inplace {NOTEBOOK_NAME}

print(f"Cleaned {NOTEBOOK_NAME}")
from google.colab import files
files.download(NOTEBOOK_NAME)